# Text to speech with `indic-speak`

Natural speech in 22 Indian languages plus English, 45 voices, with optional speaking styles.

| | |
|---|---|
| Endpoint | `POST /v1/audio/speech` (JSON, OpenAI-compatible) |
| Model | `indic-speak` |
| Input | `input` text, `voice`, `instructions` (a **JSON string** with `lang` and optional `style`) |
| Output | `audio/wav`, PCM16, 24 kHz, mono |
| Length | a sentence or two per request (~30 s of speech); chunk longer text |

In [ ]:
%pip install -q requests==2.32.3

In [ ]:
import os, json, requests

BASE_URL = os.environ.get("BODHAN_BASE_URL", "https://api.bodhan.ai")
API_KEY = os.environ["BODHAN_API_KEY"]  # export BODHAN_API_KEY=... before starting Jupyter
HEADERS = {"Authorization": f"Bearer {API_KEY}"}


def raise_for_bodhan(resp):
    """Bodhan errors are JSON: {"error": {"message", "code", "request_id"}}. Surface them readably."""
    if resp.ok:
        return resp
    try:
        err = resp.json()["error"]
        raise RuntimeError(f"{resp.status_code} {err.get('code')}: {err.get('message')} (request_id={err.get('request_id')})")
    except (ValueError, KeyError):
        resp.raise_for_status()

## 1. Speak a sentence

Note that `instructions` is a JSON *string*, not a nested object.

In [ ]:
from pathlib import Path

OUT = Path("outputs"); OUT.mkdir(exist_ok=True)


def speak(text: str, voice: str, lang: str, style: str | None = None, out: str | Path = OUT / "speech.wav") -> Path:
    instructions = {"lang": lang}
    if style:
        instructions["style"] = style
    resp = requests.post(
        f"{BASE_URL}/v1/audio/speech",
        headers={**HEADERS, "Content-Type": "application/json"},
        json={"model": "indic-speak", "input": text, "voice": voice, "instructions": json.dumps(instructions)},
        timeout=120,
    )
    raise_for_bodhan(resp)
    Path(out).write_bytes(resp.content)
    return Path(out)


path = speak("कक्षा नौ बजे शुरू होती है।", voice="Kavya", lang="hi")
print(path, path.stat().st_size, "bytes")

In [ ]:
from IPython.display import Audio
Audio(str(path))

## 2. Voices

Pick a voice that matches `lang`. Two per language (one female, one male); Hindi has three.

| Lang | Female | Male | Lang | Female | Male |
|---|---|---|---|---|---|
| `as` | Prastuti | Ankur | `mni` | Thoibi | Chaoba |
| `bn` | Ishita | Sourav | `mr` | Anagha | Chinmay |
| `brx` | Gwrbw | Sansuma | `ne` | Srijana | Sagar |
| `doi` | Preeti | Sham | `or` | Itishree | Akash |
| `gu` | Dhara | Parth | `pa` | Kaur | Manpreet |
| `hi` | Kavya, Suhani | Amit | `sa` | Bharati | Aryaman |
| `kn` | Deepika | Adarsh | `sat` | Phulmani | Sibu |
| `kok` | Anjali | Sandeep | `sd` | Moomal | Rano |
| `ks` | Zoon | Ishfaq | `ta` | Anitha | Arun |
| `mai` | Vaidehi | Madhukar | `te` | Sravani | Vamsi |
| `ml` | Lakshmi | Kiran | `ur` | Saba | Zaid |

The full list also lives in [`scripts/bodhan_api_rules.json`](../../scripts/bodhan_api_rules.json).

In [ ]:
samples = [
    ("ta", "Anitha", "வணக்கம், இன்று வகுப்பு பத்து மணிக்கு."),
    ("bn", "Sourav", "আগামীকাল স্কুল বন্ধ থাকবে।"),
    ("en", "Amit", "Welcome to the Bodhan AI cookbook."),
]
for lang, voice, text in samples:
    print(speak(text, voice=voice, lang=lang, out=OUT / f"sample_{lang}.wav"))

## 3. Styles

`style` inside `instructions` shapes delivery. Available values:

`AIR style news`, `TV style news`, `news`, `Customer Care`, `advertisements`, `educational lecture`, `children's stories`, `single person narration audiobook`, `happy`, `sad`, `anger`, `fear`, `disgust`, `surprise`

In [ ]:
line = "आज का मुख्य समाचार: राज्य में नई शिक्षा नीति लागू हो गई है।"
for style in ("TV style news", "children's stories", "educational lecture"):
    print(style, "->", speak(line, voice="Suhani", lang="hi", style=style, out=OUT / f"style_{style.replace(' ', '_')}.wav"))

## 4. Longer text: split into sentences and join the WAVs

Each request should carry a sentence or two. Split on sentence terminators (`।`, `.`, `?`, `!`), synthesise each, and concatenate the PCM frames. All clips share the same format (PCM16, 24 kHz, mono), so joining is a byte-level append.

In [ ]:
import re, wave, io, contextlib

SENTENCE_END = re.compile(r"(?<=[।.!?])\s+")


def speak_long(text: str, voice: str, lang: str, out: Path, **kw) -> Path:
    sentences = [s for s in SENTENCE_END.split(text.strip()) if s]
    frames, params = [], None
    for i, sentence in enumerate(sentences):
        clip = speak(sentence, voice, lang, out=OUT / f"_part{i}.wav", **kw)
        with contextlib.closing(wave.open(str(clip), "rb")) as wf:
            params = params or wf.getparams()
            frames.append(wf.readframes(wf.getnframes()))
    with contextlib.closing(wave.open(str(out), "wb")) as wf:
        wf.setparams(params)
        for f in frames:
            wf.writeframes(f)
    return out


story = "एक गाँव में एक किसान रहता था। उसके पास एक सुनहरा अंडा देने वाली मुर्गी थी। हर दिन वह एक अंडा बेचता था।"
speak_long(story, voice="Amit", lang="hi", style="children's stories", out=OUT / "story.wav")

**Next:** build a read-aloud pipeline: [`translate`](../translate/translate.ipynb) a notice, then `speak` it in each language. See [`examples/`](../../examples/) for complete projects.